# Lógica de Limpieza — Dataset Plantas Generadoras de Oxígeno

Este documento explica qué se hizo en cada paso del pipeline de limpieza, por qué se hizo y qué se eliminó o transformó.

---

## Contexto del dataset

Los datos provienen de plantas generadoras de oxígeno in situ (proceso PSA) instaladas en pontones salmoneros. Cada fila es una medición por minuto de un pontón (`source`). Las plantas tienen dos arquitecturas posibles:

| Arquitectura | Compresores | PSA | Jaulas máx. |
|---|---|---|---|
| Tipo A | 4 | 4 | Hasta 12 |
| Tipo B | 3 | 6 | Hasta 12 |

Cada jaula puede tener hasta 12 sensores (`s1`–`s12`), pero una planta con 8 jaulas físicas no tiene sensores `s9`–`s12`.

---

## PASO 1 — Eliminar columnas basura

**Qué:** Se eliminan `Unnamed: 0`, `Unnamed: 0.1` y `Sistema`.

**Por qué:** Son artefactos del proceso de exportación a CSV (índices guardados por error). No aportan información y ocupan memoria.

---

## PASO 2 — Eliminar pontones con arquitectura inválida

**Qué:** Se eliminan las filas de `POX1`, `POX47`, `POX64`, `POX66` y `POX67`.

**Por qué:** El análisis exploratorio reveló que estos pontones presentan una arquitectura imposible según el experto del dominio (3 compresores + 4 PSA no existe), o bien enviaban señales "fantasma" continuas en `hb_com4` (un compresor que reporta estar vivo sin existir). POX1 era el único bien programado de ese grupo pero también cae en la categoría inválida por arquitectura.

---

## PASO 3 — Compresión de tipos de dato

**Qué:** Las columnas `float64` se convierten a tipos más livianos:
- Columnas `r_`, `m_`, `hb_`, `rst_`, `suma_compresores` → `Int8` (valores esperados: 0 y 1)
- Resto de métricas continuas (presión, flujo, oxígeno) → `float32`

**Por qué:** El dataset tiene cientos de miles de filas. Sin compresión, se satura la RAM. `float32` mantiene suficiente precisión para estos sensores industriales.

---

## PASO 4 — Separar en dos dataframes según arquitectura

**Qué:** Se separa `df` en `df_4comp` y `df_3comp`.

**Cómo se detecta:** Si un `source` tiene al menos un valor no nulo en `r_com4`, tiene 4 compresores. Si `r_com4` es siempre nulo, tiene 3 compresores.

**Por qué:** Las dos arquitecturas tienen distinto número de PSA y columnas relevantes. Mezclarlas generaría ruido masivo de nulos estructurales.

---

## PASO 5 — Purgar columnas que no aplican a cada arquitectura

**Qué:**
- `df_4comp`: se eliminan todas las columnas con `psa5` o `psa6` en el nombre.
- `df_3comp`: se eliminan todas las columnas con `com4` en el nombre.

**Por qué:** Esas columnas son 100% nulas en cada subconjunto porque el hardware correspondiente no existe en esa arquitectura. Mantenerlas contamina cualquier análisis posterior.

---

## PASO 6 — Manejo de apagones totales

**Qué:** Una fila es un "apagón total" cuando **todos** los sensores de oxígeno (`ox_s1`–`ox_s12`) son nulos al mismo tiempo. Esas filas se eliminan.

**Antes de eliminarlas:** Se calcula la columna `horas_desde_mantencion`, que indica cuántas horas han pasado desde el último apagón. Esto convierte los eventos de mantenimiento en una feature útil para modelos predictivos.

**Por qué eliminar los apagones:** Durante un apagón no hay medición real de ningún sensor. Incluir esas filas sesgaría el entrenamiento de cualquier modelo.

**Qué pasa con los nulos restantes en ox_:** Después de quitar los apagones, si un sensor de oxígeno individual sigue en nulo, se marca como `-1`. Esto se distinguirá correctamente en el paso 8.

---

## PASO 7 — Reparación de telemetría de sala de máquinas

**Qué:** Para los sensores de máquina (presión, estados de compresores, PSA, etc.), se aplica:
1. **`ffill` con `limit=15`**: se propaga el último valor válido hasta 15 minutos hacia adelante.
2. **`fillna(-1)`**: lo que queda nulo después de los 15 minutos se marca como `-1`.

**Por qué:** Los cortes de telemetría breves (caída de red de segundos o minutos) no significan que la máquina paró — solo que no llegó el dato. Propagar 15 minutos es una suposición conservadora razonable para maquinaria industrial. Cortes mayores se marcan como `-1` para que el modelo entienda "desconexión prolongada".

---

## PASO 8 — Corrección maestra de jaulas y temperatura

### Jaulas: distinguir dos casos con `ox_sX = -1`

Este es el punto más crítico de la limpieza. Ambos casos producen `ox_sX = -1`, pero tienen significados opuestos:

| Caso | Descripción | `ox_sX` | `hb_sX` | `sp_sX` | `m_sX` |
|---|---|---|---|---|---|
| **Jaula inexistente** | El sensor físicamente no existe en ese pontón (ej. planta de 8 jaulas, sensor 10 no existe) | -1 | **-1** | -1 | -1 |
| **Sensor fallido** | La jaula existe pero el sensor se desconectó temporalmente | -1 | **0** | -1 | -1 |

**Cómo se detecta cuál es cuál:** Si el **máximo histórico** de `ox_sX` en ese pontón nunca superó 0, la jaula nunca existió → es fantasma. Si en algún momento tuvo valores reales, el `-1` actual es una falla temporal.

**Por qué importa la diferencia en `hb_sX`:**
- `hb = -1`: el slot no existe en el hardware. Es una señal de "ausencia estructural".
- `hb = 0`: el equipo existe pero está offline. Un modelo puede aprender que "existía y se cayó" es diferente a "nunca estuvo ahí".

### Temperatura: códigos de error de hardware

Algunos PLCs envían `-32768` o `9999` como código de error cuando el sensor de temperatura falla. Estos valores se detectan con el filtro `< -100` o `> 1000` y se reemplazan por el último valor válido (`ffill` por pontón).

---

## PASO 9 — Saneamiento de heartbeats con valores anómalos

**Qué:** Algunos PLCs enviaban valores numéricos extraños en columnas `hb_` (ej. `17307`, `2026`). Se normalizan a `0` (muerto/desconectado).

**Importante:** Los `-1` legítimos (jaulas inexistentes, asignados en el paso 8) **no se tocan**. Solo se reemplazan los valores fuera del conjunto `{0, 1, -1}`.

**Por qué no se usan como `1` (vivo):** No se puede asumir que un valor anómalo significa "activo". La política conservadora es tratarlo como desconocido → `0`.

---

## PASO 10 — Auditoría final

Se verifica que:
- Todas las columnas `hb_` contienen solo `{0, 1, -1}` (**incluyendo `-1`** como válido, a diferencia del notebook original que solo aceptaba `{0, 1}`)
- No quedan nulos en sensores de oxígeno
- No quedan nulos en temperatura
- Nulos totales = 0

---

## Resultado final

| Dataset | Descripción |
|---|---|
| `dataset_4comp_limpio.csv` | Pontones de 4 compresores y 4 PSA, sin columnas de PSA5/6 |
| `dataset_3comp_limpio.csv` | Pontones de 3 compresores y 6 PSA, sin columnas de COM4 |

### Convención de valores especiales

| Valor | Significado |
|---|---|
| `0` | Equipo apagado / sensor offline |
| `1` | Equipo encendido / sensor activo |
| `-1` | Jaula/equipo inexistente en este pontón, O corte de telemetría largo (>15 min) |
| `9999` en `horas_desde_mantencion` | El pontón nunca ha tenido un apagón registrado |


# 🧹 limpieza
Pipeline de limpieza para el dataset de plantas generadoras de oxígeno.
Basado en el análisis exploratorio de `analisis_dataset.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import gc

ruta_csv = 'global_concatenado.CSV'

print('Cargando dataset...')
df = pd.read_csv(ruta_csv)
print(f'Filas: {len(df):,} | Columnas: {df.shape[1]}')

## PASO 1 — Eliminar columnas basura

In [ ]:
columnas_basura = ['Unnamed: 0.1', 'Unnamed: 0', 'Sistema']
df.drop(columns=[c for c in columnas_basura if c in df.columns], inplace=True)
print(f'✅ Columnas basura eliminadas. Shape: {df.shape}')

: 

## PASO 2 — Eliminar pontones con arquitectura inválida
Los pontones POX1, POX47, POX64, POX66 y POX67 reportaron arquitecturas
imposibles (3 compresores + 4 PSA) o señales fantasma persistentes en hb_com4.

In [5]:
pontones_invalidos = ['POX1', 'POX47', 'POX64', 'POX66', 'POX67']
indices_drop = df[df['source'].isin(pontones_invalidos)].index
df.drop(index=indices_drop, inplace=True)
print(f'✅ {len(indices_drop):,} filas de pontones inválidos eliminadas. Shape: {df.shape}')

✅ 785,912 filas de pontones inválidos eliminadas. Shape: (7858280, 106)


## PASO 3 — Compresión de tipos de dato

In [6]:
print('Comprimiendo columnas...')
for col in df.columns:
    if df[col].dtype == 'float64':
        if col.startswith(('r_', 'm_', 'hb_', 'rst_')) or col == 'suma_compresores':
            mn, mx = df[col].min(), df[col].max()
            if pd.isna(mx) or (mn >= -128 and mx <= 127):
                df[col] = df[col].astype('Int8')
            else:
                df[col] = df[col].astype('float32')
        else:
            df[col] = df[col].astype('float32')
gc.collect()
print('✅ Compresión completada.')

Comprimiendo columnas...
✅ Compresión completada.


## PASO 4 — Separar en dos dataframes según arquitectura
- **4 compresores → 4 PSA**: detectados porque tienen datos en `r_com4`
- **3 compresores → 6 PSA**: el resto

In [7]:
sources_4comp = df.dropna(subset=['r_com4'])['source'].unique()

df_4comp = df[df['source'].isin(sources_4comp)].copy()
df_3comp = df[~df['source'].isin(sources_4comp)].copy()

del df
gc.collect()

print(f'✅ 4 Compresores: {len(df_4comp):,} filas')
print(f'✅ 3 Compresores: {len(df_3comp):,} filas')

✅ 4 Compresores: 5,862,409 filas
✅ 3 Compresores: 1,995,871 filas


## PASO 5 — Purga de columnas que no corresponden a cada arquitectura
- df_4comp: no tiene PSA 5 ni PSA 6 → se eliminan
- df_3comp: no tiene compresor 4 → se eliminan

In [8]:
cols_psa56  = [c for c in df_4comp.columns if 'psa5' in c or 'psa6' in c]
cols_com4   = [c for c in df_3comp.columns if 'com4' in c]

df_4comp.drop(columns=cols_psa56, inplace=True, errors='ignore')
df_3comp.drop(columns=cols_com4,  inplace=True, errors='ignore')

print(f'✅ df_4comp: {df_4comp.shape[1]} columnas (eliminadas {len(cols_psa56)} de PSA5/6)')
print(f'✅ df_3comp: {df_3comp.shape[1]} columnas (eliminadas {len(cols_com4)} de COM4)')

✅ df_4comp: 98 columnas (eliminadas 8 de PSA5/6)
✅ df_3comp: 103 columnas (eliminadas 3 de COM4)


## PASO 6 — Manejo de apagones totales
Cuando TODOS los sensores de oxígeno son nulos a la vez, es un apagón
de red (mantenimiento o corte de internet). Antes de eliminar esas filas,
se calcula `horas_desde_mantencion` para conservar esa info temporalmente.

In [9]:
cols_ox = [f'ox_s{i}' for i in range(1, 13)]

def crear_memoria_temporal(df, nombre):
    print(f'⏳ {nombre}...')
    df['TIME'] = pd.to_datetime(df['TIME'], errors='coerce')
    df.sort_values(by=['source', 'TIME'], inplace=True)

    cols_ox_presentes = [c for c in cols_ox if c in df.columns]
    df['es_apagon'] = df[cols_ox_presentes].isnull().all(axis=1)
    df.loc[df['es_apagon'], 'tiempo_apagon'] = df['TIME']
    df['ultimo_apagon'] = df.groupby('source')['tiempo_apagon'].ffill()

    diff_seg = (df['TIME'] - df['ultimo_apagon']).dt.total_seconds()
    df['horas_desde_mantencion'] = (diff_seg / 3600.0).fillna(9999.0).astype('float32')

    filas_antes = len(df)
    df.drop(df[df['es_apagon']].index, inplace=True)
    df[cols_ox_presentes] = df[cols_ox_presentes].fillna(-1)
    df.drop(columns=['es_apagon', 'tiempo_apagon', 'ultimo_apagon'], inplace=True)

    print(f'  ✅ Apagones eliminados: {filas_antes - len(df):,} filas')
    return df

df_4comp = crear_memoria_temporal(df_4comp, 'Sistema 4 Compresores')
df_3comp = crear_memoria_temporal(df_3comp, 'Sistema 3 Compresores')

⏳ Sistema 4 Compresores...
  ✅ Apagones eliminados: 312,709 filas
⏳ Sistema 3 Compresores...
  ✅ Apagones eliminados: 8,645 filas


## PASO 7 — Reparación de telemetría de sala de máquinas
Cortes de ≤15 minutos en sensores de máquina (psi, r_, etc): se propaga
el último valor válido (ffill). Cortes mayores: se marca como -1.

In [10]:
def reparar_telemetria(df, nombre):
    print(f'⏳ {nombre}...')
    df.sort_values(by=['source', 'TIME'], inplace=True)

    cols_ox_presentes = [c for c in cols_ox if c in df.columns]
    cols_excluir = ['source', 'TIME', 'horas_desde_mantencion'] + cols_ox_presentes
    cols_maquina = df.columns.drop(cols_excluir)

    # Cortes cortos: propagar hasta 15 minutos
    df[cols_maquina] = df.groupby('source')[cols_maquina].ffill(limit=15)
    # Cortes largos: marcar como -1
    df[cols_maquina] = df[cols_maquina].fillna(-1)

    print(f'  ✅ Nulos restantes: {df.isnull().sum().sum()}')
    return df

df_4comp = reparar_telemetria(df_4comp, 'Sistema 4 Compresores')
df_3comp = reparar_telemetria(df_3comp, 'Sistema 3 Compresores')

⏳ Sistema 4 Compresores...
  ✅ Nulos restantes: 0
⏳ Sistema 3 Compresores...
  ✅ Nulos restantes: 0


## PASO 8 — Corrección maestra de jaulas y temperatura
Distingue dos casos que ambos tienen `ox_sX = -1`:
- **Jaula inexistente** (nunca tuvo valor real): `ox=-1, hb=-1, sp=-1, m=-1`
- **Sensor fallido temporalmente** (la jaula sí existe): `ox=-1, hb=0, sp=-1, m=-1`

También limpia códigos de error de hardware en temperatura (<-100 o >1000).

In [11]:
def correccion_maestra(df, nombre):
    print(f'⏳ {nombre}...')

    # --- Temperatura ---
    for col in ['mb_g1_temperatura_f', 'mb_g2_temperatura_f']:
        if col in df.columns:
            df.loc[(df[col] < -100) | (df[col] > 1000), col] = np.nan
            df[col] = df.groupby('source')[col].ffill().fillna(0)

    # --- Jaulas ---
    for i in range(1, 13):
        col_ox = f'ox_s{i}'
        col_hb = f'hb_s{i}'
        col_sp = f'sp_s{i}'
        col_m  = f'm_s{i}'
        if col_ox not in df.columns:
            continue

        max_ox = df.groupby('source')[col_ox].transform('max')

        # Caso A: Jaula INEXISTENTE (máximo histórico nunca superó 0)
        fantasma = (max_ox <= 0)
        for col in [col_ox, col_hb, col_sp, col_m]:
            if col in df.columns:
                df.loc[fantasma, col] = -1

        # Caso B: Sensor FALLIDO (la jaula existe pero ox cayó a -1 temporalmente)
        falla = (~fantasma) & (df[col_ox] == -1)
        for col in [col_sp, col_m]:
            if col in df.columns:
                df.loc[falla, col] = -1
        if col_hb in df.columns:
            df.loc[falla, col_hb] = 0   # muerto pero existente ≠ -1

    print('  ✅ Jaulas y temperatura corregidas.')
    return df

df_4comp = correccion_maestra(df_4comp, 'Sistema 4 Compresores')
df_3comp = correccion_maestra(df_3comp, 'Sistema 3 Compresores')

⏳ Sistema 4 Compresores...
  ✅ Jaulas y temperatura corregidas.
⏳ Sistema 3 Compresores...
  ✅ Jaulas y temperatura corregidas.


## PASO 9 — Saneamiento de heartbeats con valores anómalos
Algunos PLCs envían valores raros (ej. 17307, 2026) en columnas `hb_`.
Se normalizan a 0 (muerto). Los -1 legítimos (jaulas inexistentes) se conservan.

In [12]:
def saneamiento_hb(df, nombre):
    print(f'⏳ {nombre}...')
    cols_hb = [c for c in df.columns if c.startswith('hb_')]
    for col in cols_hb:
        # Solo los valores fuera de {0, 1, -1} son ruido
        mascara = ~df[col].isin([0.0, 1.0, -1.0])
        df.loc[mascara, col] = np.nan
        df[col] = df[col].fillna(0).astype('Int8')
    print(f'  ✅ {len(cols_hb)} columnas hb_ saneadas.')
    return df

df_4comp = saneamiento_hb(df_4comp, 'Sistema 4 Compresores')
df_3comp = saneamiento_hb(df_3comp, 'Sistema 3 Compresores')

⏳ Sistema 4 Compresores...
  ✅ 24 columnas hb_ saneadas.
⏳ Sistema 3 Compresores...
  ✅ 25 columnas hb_ saneadas.


## PASO 10 — Auditoría final
Verifica que no queden nulos ni valores fuera de rango esperado.

In [13]:
def auditoria_final(df, nombre):
    print(f'\n--- Auditoría Final: {nombre} ---')
    errores = 0

    # hb_ solo puede tener 0, 1, -1  (-1 es jaula inexistente, es válido)
    for col in [c for c in df.columns if c.startswith('hb_')]:
        invalidos = df[~df[col].isin([0, 1, -1])][col].unique()
        if len(invalidos) > 0:
            print(f'  ⚠️  {col}: valores inválidos {invalidos}')
            errores += 1

    cols_ox_presentes = [c for c in [f'ox_s{i}' for i in range(1,13)] if c in df.columns]
    nulos_ox   = df[cols_ox_presentes].isnull().sum().sum()
    nulos_temp = df[['mb_g1_temperatura_f','mb_g2_temperatura_f']].isnull().sum().sum()
    nulos_total = df.isnull().sum().sum()

    print(f'  Nulos en ox_s*:      {nulos_ox}')
    print(f'  Nulos en temperatura: {nulos_temp}')
    print(f'  Nulos totales:        {nulos_total}')

    if errores == 0 and nulos_total == 0:
        print('  ✅ Dataset limpio.')
    else:
        print(f'  ❌ Se encontraron problemas ({errores} alertas de hb, {nulos_total} nulos).')

auditoria_final(df_4comp, 'Sistema 4 Compresores')
auditoria_final(df_3comp, 'Sistema 3 Compresores')


--- Auditoría Final: Sistema 4 Compresores ---
  Nulos en ox_s*:      0
  Nulos en temperatura: 0
  Nulos totales:        0
  ✅ Dataset limpio.

--- Auditoría Final: Sistema 3 Compresores ---
  Nulos en ox_s*:      0
  Nulos en temperatura: 0
  Nulos totales:        0
  ✅ Dataset limpio.


## PASO 11 — Guardar resultados

In [14]:
df_4comp.to_csv('dataset_4comp_limpio.csv', index=False)
df_3comp.to_csv('dataset_3comp_limpio.csv', index=False)

print(f'✅ dataset_4comp_limpio.csv  → {df_4comp.shape}')
print(f'✅ dataset_3comp_limpio.csv  → {df_3comp.shape}')


✅ dataset_4comp_limpio.csv  → (5549700, 99)
✅ dataset_3comp_limpio.csv  → (1987226, 104)


In [15]:
print("datos totales", df_4comp.shape[0] + df_3comp.shape[0])

datos totales 7536926


In [16]:
print("Columnas df_4comp:", df_4comp.columns)
print("Columnas df_3comp:", df_3comp.columns)

Columnas df_4comp: Index(['source', 'psi_psa1', 'psi_psa2', 'psi_psa3', 'psi_psa4', 'psi_tablero',
       'flujo', 'totalizador', 'TIME', 'r_psa1', 'r_psa2', 'r_psa3', 'r_psa4',
       'r_gen1', 'r_gen2', 'r_bar', 'r_sec1', 'r_sec2', 'r_com1', 'r_com2',
       'r_com3', 'r_com4', 'sp_s1', 'sp_s2', 'sp_s3', 'sp_s4', 'sp_s5',
       'sp_s6', 'sp_s7', 'sp_s8', 'sp_s9', 'sp_s10', 'sp_s11', 'sp_s12',
       'hb_gen1', 'hb_gen2', 'hb_sec1', 'hb_sec2', 'hb_com1', 'hb_com2',
       'hb_com3', 'hb_com4', 'hb_psa1', 'hb_psa2', 'hb_psa3', 'hb_psa4',
       'rst_gen1', 'rst_gen2', 'rst_sec1', 'rst_sec2', 'rst_com1', 'rst_com2',
       'rst_com3', 'rst_com4', 'rst_psa1', 'rst_psa2', 'rst_psa3', 'rst_psa4',
       'hb_s1', 'hb_s2', 'hb_s3', 'hb_s4', 'hb_s5', 'hb_s6', 'hb_s7', 'hb_s8',
       'hb_s9', 'hb_s10', 'hb_s11', 'hb_s12', 'ox_s1', 'ox_s2', 'ox_s3',
       'ox_s4', 'ox_s5', 'ox_s6', 'ox_s7', 'ox_s8', 'ox_s9', 'ox_s10',
       'ox_s11', 'ox_s12', 'm_s1', 'm_s2', 'm_s3', 'm_s4', 'm_s5', 'm_s6',